# Sesión 01 — Modelado del Canal Inalámbrico
### Laboratorio Python · Sistemas de Comunicaciones Inalámbricas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/01-channel-modeling/lab.ipynb)

---
> **Licencia**: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) · *Sistemas de Comunicaciones Inalámbricas* por ollerenac

## Objetivos del Laboratorio

Al completar este laboratorio serás capaz de:

1. Simular la envolvente de desvanecimiento Rayleigh usando el modelo de Clarke y verificar la distribución teórica.
2. Comparar distribuciones de amplitud para desvanecimiento Rician con distintos valores del factor K.
3. Calcular y representar la curva de BER frente a SNR para BPSK sobre canal Rayleigh y compararla con el canal AWGN.
4. Interpretar la degradación de rendimiento debida al desvanecimiento.

In [ ]:
%pip install numpy scipy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rayleigh, rice
from scipy.special import i0, i0e

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
rng = np.random.default_rng(seed=42)

## Repaso Teórico

### Respuesta al impulso del canal multitrayecto

El canal multitrayecto banda estrecha se modela como un coeficiente complejo:

$$h = h_I + j\,h_Q$$

donde $h_I$ y $h_Q$ son componentes en fase y cuadratura.

### Desvanecimiento Rayleigh

Cuando no hay componente dominante (LOS), $h_I, h_Q \sim \mathcal{N}(0, \sigma^2)$ y la envolvente $r = |h|$ sigue una distribución Rayleigh:

$$f_R(r) = \frac{r}{\sigma^2}\,e^{-r^2/(2\sigma^2)}, \quad r \geq 0$$

La BER de BPSK en canal Rayleigh plano (promediada sobre el desvanecimiento) es:

$$P_e = \frac{1}{2}\left(1 - \sqrt{\frac{\bar{\gamma}}{1+\bar{\gamma}}}\right)$$

donde $\bar{\gamma} = E_b/N_0$ es la SNR media por bit.

### Desvanecimiento Rician

Cuando existe una componente especular dominante con amplitud $\nu$, la envolvente sigue una distribución Rice:

$$f_R(r) = \frac{r}{\sigma^2}\,e^{-(r^2+\nu^2)/(2\sigma^2)}\,I_0\!\left(\frac{r\nu}{\sigma^2}\right), \quad r \geq 0$$

El **factor K** es la razón entre la potencia de la componente especular y la potencia difusa:

$$K = \frac{\nu^2}{2\sigma^2}$$

$K=0$ corresponde a Rayleigh puro; $K\to\infty$ equivale a canal AWGN sin desvanecimiento.

---
## Ejercicio 1 — Desvanecimiento Rayleigh: simulación y verificación de la PDF

Generaremos las componentes en fase y cuadratura de un canal Rayleigh plano y comprobaremos que la envolvente simulada coincide con la PDF teórica de Rayleigh.

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
N = 100_000          # número de muestras del canal
sigma = 1 / np.sqrt(2)  # desviación típica de cada componente → E[r²] = 1

# ── Simulación (modelo de Clarke simplificado) ───────────────────────────────
h_I = rng.normal(0, sigma, N)
h_Q = rng.normal(0, sigma, N)
r = np.abs(h_I + 1j * h_Q)   # envolvente

# ── PDF teórica de Rayleigh con σ=1/√2 ──────────────────────────────────────
r_axis = np.linspace(0, 4, 500)
pdf_theory = rayleigh.pdf(r_axis, scale=sigma)

# ── Figura ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Serie temporal de la envolvente (primeras 500 muestras)
axes[0].plot(r[:500], lw=0.8, color='steelblue')
axes[0].set_xlabel('Índice de muestra')
axes[0].set_ylabel('Envolvente $r$')
axes[0].set_title('Envolvente Rayleigh — primeras 500 muestras')

# (b) Histograma vs PDF teórica
axes[1].hist(r, bins=80, density=True, alpha=0.6, color='steelblue', label='Simulada')
axes[1].plot(r_axis, pdf_theory, 'r-', lw=2, label='Teórica Rayleigh')
axes[1].set_xlabel('Envolvente $r$')
axes[1].set_ylabel('Densidad de probabilidad')
axes[1].set_title('PDF de la envolvente Rayleigh')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Potencia media simulada : {np.mean(r**2):.4f}  (esperada: {2*sigma**2:.4f})')
print(f'Valor cuadrático medio  : {np.sqrt(np.mean(r**2)):.4f}')

**Preguntas de reflexión:**

1. ¿Por qué la envolvente nunca es negativa aunque $h_I$ y $h_Q$ sí lo son?
2. ¿Qué ocurre con la forma de la PDF si aumentas $\sigma$? Prueba con $\sigma = 0.5$ y $\sigma = 2$.
3. ¿Cuál es la probabilidad de que la envolvente caiga por debajo de $-10\,\text{dB}$ (es decir, $r < 10^{-10/20}$)? Calcula con la CDF teórica.

---
## Ejercicio 2 — Desvanecimiento Rician: efecto del factor K

Simularemos y compararemos las PDFs de la envolvente para K = 0, 1, 5, 10 dB.

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
N = 200_000
K_dB_list = ['-inf (Rayleigh)', '0 dB', '5 dB', '10 dB']
K_lin_list = [0, 1, 10**0.5, 10]          # K en lineal
colors = ['steelblue', 'darkorange', 'green', 'red']

r_axis = np.linspace(0, 3.5, 500)

fig, ax = plt.subplots(figsize=(9, 5))

for K, label, color in zip(K_lin_list, K_dB_list, colors):
    # Potencia total unitaria: ν² + 2σ² = 1  →  σ² = 1/(2(K+1)), ν² = K/(K+1)
    sigma2 = 1 / (2 * (K + 1))
    sigma  = np.sqrt(sigma2)
    nu     = np.sqrt(K / (K + 1))   # amplitud de la componente especular

    # Simulación: añadimos componente LOS a las gaussianas
    h_I = rng.normal(nu, sigma, N)
    h_Q = rng.normal(0,  sigma, N)
    r_sim = np.abs(h_I + 1j * h_Q)

    # PDF teórica de Rice usando scipy (b = ν/σ, scale = σ)
    pdf_th = rice.pdf(r_axis, b=nu/sigma if sigma > 0 else 0, scale=sigma)

    ax.hist(r_sim, bins=100, density=True, alpha=0.25, color=color)
    ax.plot(r_axis, pdf_th, color=color, lw=2, label=f'K = {label}')

ax.set_xlabel('Envolvente $r$')
ax.set_ylabel('Densidad de probabilidad')
ax.set_title('Distribución de Rice para distintos valores de K')
ax.legend()
plt.tight_layout()
plt.show()

**Preguntas de reflexión:**

1. A medida que K aumenta, la PDF se desplaza hacia la derecha y se estrecha. ¿Qué implica esto para la probabilidad de desvanecimieno profundo?
2. Para $K=0$, verifica que la distribución Rice se reduce a Rayleigh.
3. Un enlace en interiores sin LOS tiene $K \approx 0$; un enlace satélite puede tener $K > 20\,\text{dB}$. ¿Qué consecuencia tiene esto en el diseño del margen de enlace?

---
## Ejercicio 3 — Curvas BER vs SNR: Rayleigh frente a AWGN

Calcularemos y compararemos la BER de BPSK en:
- Canal AWGN: $P_e^{\text{AWGN}} = Q(\sqrt{2\,E_b/N_0})$
- Canal Rayleigh plano (expresión cerrada): $P_e^{\text{Ray}} = \frac{1}{2}\left(1-\sqrt{\frac{\bar{\gamma}}{1+\bar{\gamma}}}\right)$
- Canal Rayleigh plano (simulación Monte Carlo)

In [ ]:
from scipy.special import erfc

# ── Rango de SNR ─────────────────────────────────────────────────────────────
EbN0_dB  = np.arange(0, 31, 2)
EbN0_lin = 10 ** (EbN0_dB / 10)

# ── BER teórica AWGN ─────────────────────────────────────────────────────────
Q = lambda x: 0.5 * erfc(x / np.sqrt(2))
ber_awgn = Q(np.sqrt(2 * EbN0_lin))

# ── BER teórica Rayleigh (forma cerrada) ─────────────────────────────────────
ber_rayleigh_th = 0.5 * (1 - np.sqrt(EbN0_lin / (1 + EbN0_lin)))

# ── Simulación Monte Carlo sobre canal Rayleigh ───────────────────────────────
N_bits = 200_000
bits   = rng.integers(0, 2, N_bits)          # bits aleatorios
s      = 2 * bits - 1                         # BPSK: {0,1} → {-1,+1}

ber_mc = np.zeros(len(EbN0_dB))
sigma_channel = 1 / np.sqrt(2)               # canal Rayleigh con potencia unitaria

for idx, snr in enumerate(EbN0_lin):
    # Coeficiente de canal complejo (Rayleigh)
    h = (rng.normal(0, sigma_channel, N_bits)
         + 1j * rng.normal(0, sigma_channel, N_bits))
    # Potencia de ruido AWGN
    sigma_n = np.sqrt(0.5 / snr)
    noise   = sigma_n * rng.normal(0, 1, N_bits)
    # Señal recibida y detección con conocimiento perfecto del canal (coherente)
    y       = np.real(h) * s + noise
    bits_rx = (y > 0).astype(int)
    ber_mc[idx] = np.mean(bits_rx != bits)

# ── Figura ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(EbN0_dB, ber_awgn,       'b-o',  lw=2, ms=6, label='AWGN (teórica)')
ax.semilogy(EbN0_dB, ber_rayleigh_th,'r-s',  lw=2, ms=6, label='Rayleigh (teórica)')
ax.semilogy(EbN0_dB, ber_mc,         'g--^', lw=1.5, ms=6, label='Rayleigh (Monte Carlo)')

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('BER vs $E_b/N_0$ — BPSK sobre AWGN y canal Rayleigh plano')
ax.set_ylim([1e-5, 1])
ax.legend()
plt.tight_layout()
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────────
print(f'{"Eb/N0 (dB)":>12} {"AWGN":>12} {"Rayleigh (th)":>15} {"Rayleigh (MC)":>15}')
print('-' * 58)
for i, db in enumerate(EbN0_dB):
    print(f'{db:>12.0f} {ber_awgn[i]:>12.2e} {ber_rayleigh_th[i]:>15.2e} {ber_mc[i]:>15.2e}')

**Preguntas de reflexión:**

1. A $E_b/N_0 = 20\,\text{dB}$, ¿cuántos dB se necesitan *extra* sobre el canal Rayleigh para alcanzar la misma BER que en AWGN? Este valor se denomina **penalización por desvanecimiento**.
2. La curva de Rayleigh teórica decae como $1/(4\bar{\gamma})$ a alta SNR, mucho más lento que la gaussiana del AWGN. ¿Por qué esta diferencia fundamental en la pendiente de la curva?
3. ¿Cómo modificarías la simulación para modelar **diversidad de recepción** con dos antenas independientes (MRC, maximal-ratio combining)? Pista: promedia dos coeficientes de canal independientes ponderados por su magnitud.

---
## Conclusiones

En este laboratorio hemos:

- **Verificado** que la suma de dos gaussianas IID produce una envolvente Rayleigh, confirmando el modelo estadístico del canal.
- **Observado** cómo el factor K de Rician desplaza la distribución de la envolvente, reduciendo la probabilidad de desvanecimiento profundo a medida que K aumenta.
- **Cuantificado** la degradación severa de la BER en un canal con desvanecimiento Rayleigh frente a AWGN: la curva BER cae mucho más lentamente (pendiente 1/SNR vs exponencial), lo que exige técnicas adicionales (codificación, diversidad, MIMO) para alcanzar prestaciones aceptables.

Estos resultados motivan directamente los temas de las siguientes sesiones: OFDM (Sesión 03), codificación de canal (Sesión 04) y sistemas MIMO (Sesión 05).

---

### Recursos Adicionales

- T. S. Rappaport, *Wireless Communications*, 2nd ed. — Cap. 3 y 5
- A. Goldsmith, *Wireless Communications*, Cambridge, 2005 — Cap. 3
- scipy.stats.rayleigh: [documentación](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rayleigh.html)
- scipy.stats.rice: [documentación](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rice.html)